# Notebook 05 - Common ML Algorithms

Goes with **Lesson 07**. We train seven algorithms on the same student data.

Every algorithm uses the same 3 steps:

```python
model = SomeAlgorithm()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

## Prepare the data

In [ ]:
from pathlib import Path

# The data files are in the "data" folder next to "notebooks".
# In Google Colab, upload the CSV file and it will be found in the current folder.
DATA_DIR = Path("../data") if Path("../data").exists() else Path(".")
print("Reading data from:", DATA_DIR.resolve())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(DATA_DIR / "student_performance.csv")

# Turn text columns into numbers (Lesson 06)
data = df.drop(columns=["student_id"]).copy()
data["internet_access"] = data["internet_access"].map({"yes": 1, "no": 0})
data["extra_classes"] = data["extra_classes"].map({"yes": 1, "no": 0})
data = pd.get_dummies(data, columns=["city_type"], prefix="city", dtype=int)

# Features (X). We remove BOTH answer columns to avoid data leakage.
X = data.drop(columns=["final_score", "result"])
y_class = data["result"]        # classification label: pass / fail
y_number = data["final_score"]  # regression label: a number

print("Features used:", list(X.columns))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, yc_train, yc_test, yn_train, yn_test = train_test_split(
    X, y_class, y_number, test_size=0.2, random_state=42
)
print("Training rows:", len(X_train), " Test rows:", len(X_test))

## 1. Linear regression (predict a NUMBER: final score)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

lin = LinearRegression()
lin.fit(X_train, yn_train)

predicted = lin.predict(X_test)
print(f"On average, the prediction is off by {mean_absolute_error(yn_test, predicted):.1f} marks")
print()
print("How many marks each feature adds (the coefficients):")
coefficients = pd.Series(lin.coef_, index=X.columns).round(2).sort_values(ascending=False)
print(coefficients)

Read the first line of the table: **each extra hour of study adds about 4 marks**. Linear regression is easy to **explain**.

## 2. Logistic regression (predict a CATEGORY: pass/fail, with a probability)

KNN, SVM and logistic regression work better with **scaled** features. A **pipeline** joins the scaler and the model, so scaling happens automatically.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, yc_train)
print(f"Logistic regression accuracy: {logreg.score(X_test, yc_test):.2f}")

one_student = X_test.iloc[[0]]
probabilities = logreg.predict_proba(one_student)[0]
for label, p in zip(logreg.classes_, probabilities):
    print(f"Probability of {label}: {p:.2f}")

## 3. Decision tree (a set of yes/no questions)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, yc_train)
print(f"Decision tree accuracy: {tree.score(X_test, yc_test):.2f}")

plt.figure(figsize=(18, 8))
plot_tree(tree, feature_names=list(X.columns), class_names=list(tree.classes_), filled=True, fontsize=9)
plt.title("The questions the decision tree learned")
plt.show()

Start at the top box and follow the questions down. Each box shows the question, how many students reached it (`samples`) and the majority answer (`class`).

## 4. Random forest (100 trees that vote)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, yc_train)
print(f"Random forest accuracy: {forest.score(X_test, yc_test):.2f}")

importance = pd.Series(forest.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(7, 4))
plt.barh(importance.index, importance.values)
plt.title("Which features the forest relied on most")
plt.xlabel("Importance")
plt.show()

## 5. K-Nearest Neighbours (ask the 5 most similar students)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, yc_train)
print(f"KNN accuracy: {knn.score(X_test, yc_test):.2f}")

## 6. Support Vector Machine (the widest road between classes)

In [ ]:
from sklearn.svm import SVC

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
svm.fit(X_train, yc_train)
print(f"SVM accuracy: {svm.score(X_test, yc_test):.2f}")

## Compare the classification models

In [ ]:
models = {
    "Logistic regression": logreg,
    "Decision tree (depth 3)": tree,
    "Random forest": forest,
    "KNN (k=5)": knn,
    "SVM": svm,
}
results = pd.DataFrame(
    {"test accuracy": [m.score(X_test, yc_test) for m in models.values()]},
    index=models.keys(),
).round(3).sort_values("test accuracy", ascending=False)
results

All of them do well on this simple data. On real projects the differences are often bigger, which is why we **try several and compare**.

**Note:** with only 60 test students, one student = 1.7 percent. Small differences between models may just be luck. Lesson 08 shows a more reliable way (cross-validation).

## 7. K-means clustering (find groups WITHOUT labels)

In [ ]:
from sklearn.cluster import KMeans

habits = df[["hours_studied", "sleep_hours", "attendance_percent"]]
habits_scaled = StandardScaler().fit_transform(habits)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(habits_scaled)

print("Average habits of each group found by K-means:")
summary = df.groupby("cluster")[["hours_studied", "sleep_hours", "attendance_percent", "final_score"]].mean().round(1)
summary["students"] = df["cluster"].value_counts().sort_index()
summary

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["hours_studied"], df["attendance_percent"], c=df["cluster"], alpha=0.7)
plt.xlabel("Hours studied")
plt.ylabel("Attendance percent")
plt.title("Student groups found by K-means")
plt.show()

Look at the table and give each group a name, for example "hard workers", "low attendance", "needs support". K-means never saw `final_score`, but the groups still differ in marks.

## Try it yourself

1. In the KNN cell, change `n_neighbors=5` to `1` and to `25`.
2. In the decision tree cell, change `max_depth=3` to `10`.
3. In the K-means cell, change `n_clusters=3` to `2` or `4`.